<!-- track-identity-card -->
# Raw telemetry to columnar storage

| | |
|---|---|
| Pipeline step | `01_pkl_to_parquet.ipynb` |
| Manuscript section | - |
| Copied from | `notebooks/NB03_pkl_to_parquet_v2.ipynb` |
| Source sha256 | `3e1caee31057c45832bb74cf17f30011` |

**Reads**

- `data/raw/**/*.pkl`

**Writes**

- `data/processed/<tier>/*.parquet`

Entry point of the pipeline. Converts the pickled session dumps into parquet.

> Copied verbatim from the working notebook. The identity card above is the only addition; no code cell was modified.


# NB03 v2 — PKL → Parquet Dönüşüm (Tier Bazlı)
**T2-T4 raw verilerini processed parquet dosyalarına çevirir.**

Adım adım: T1 referansı kontrol → format doğrula → T2 → T3 → T4

## Adım 1: Konfigürasyon + Mevcut Durum

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from shutil import copy2
import warnings
warnings.filterwarnings('ignore')

PROJECT = TRACK_ROOT
DATA = PROJECT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"

HUMAN_DIR = RAW / "acgym_human"
SAC_DIR = RAW / "acgym_sac"

TIER_DIRS = {
    'T1': PROCESSED / "tier1_small_3t1c",
    'T2': PROCESSED / "tier2_medium_3t2c",
    'T3': PROCESSED / "tier3_large_3t3c",
    'T4': PROCESSED / "tier4_full_4t3c",
}
for d in TIER_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

SAC_REF = DATA / "reference_sac"
SAC_REF.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("  ADIM 1: MEVCUT DURUM")
print("=" * 65)

t1_files = list(TIER_DIRS['T1'].glob("*.parquet"))
print(f"\n  T1 mevcut: {len(t1_files)} parquet")
for f in sorted(t1_files):
    df = pd.read_parquet(f)
    print(f"    {f.name}: {len(df)} satir x {len(df.columns)} kolon")

h_pkl = list(HUMAN_DIR.rglob("*.pkl")) if HUMAN_DIR.exists() else []
h_ld = list(HUMAN_DIR.rglob("*.ld")) if HUMAN_DIR.exists() else []
s_pkl = list(SAC_DIR.rglob("*.pkl")) if SAC_DIR.exists() else []
print(f"\n  Raw insan: {len(h_pkl)} pkl + {len(h_ld)} ld")
print(f"  Raw SAC:   {len(s_pkl)} pkl")

T1_COLUMNS = None
if t1_files:
    ref_df = pd.read_parquet(t1_files[0])
    T1_COLUMNS = list(ref_df.columns)
    print(f"\n  T1 referans kolonlar ({len(T1_COLUMNS)}):")
    print(f"    {T1_COLUMNS[:10]}...")

## Adim 2: Dosyalari Tier Bazli Siniflandir

In [ ]:
print("=" * 65)
print("  ADIM 2: DOSYA SINIFLANDIRMA")
print("=" * 65)

TRACKS = ['monza', 'barcelona', 'ks_barcelona', 'red_bull_ring',
          'ks_red_bull_ring', 'indianapolis', 'indi']
CARS = ['bmw_z4_gt3', 'dallara_f317', 'ks_mazda_miata']

def classify_file(fpath):
    fpath_lower = str(fpath).lower()
    track, car = "unknown", "unknown"
    for t in TRACKS:
        if t in fpath_lower:
            if 'barcelona' in t: track = 'barcelona'
            elif 'red_bull' in t: track = 'red_bull_ring'
            elif 'indianapolis' in t or t == 'indi': track = 'indianapolis'
            else: track = t
            break
    for c in CARS:
        if c in fpath_lower:
            car = c
            break
    if car == "unknown":
        fname = str(fpath).split("/")[-1].split("\\")[-1].lower()
        if 'dallara' in fname or 'f317' in fname: car = 'dallara_f317'
        elif 'mazda' in fname or 'miata' in fname: car = 'ks_mazda_miata'
        elif 'bmw' in fname or 'z4' in fname: car = 'bmw_z4_gt3'
    return track, car

def is_sac_file(fpath):
    return any("SAC" in p.upper() for p in str(fpath).replace("\\", "/").split("/"))

def is_telemetry_file(fpath):
    f_lower = str(fpath).lower()
    skip = ['model/', '\\model', 'target_q', 'actor', 'critic', 'replay_buffer',
            'assettocorsaconfigs', '/eval/', '\\eval\\']
    return not any(x in f_lower for x in skip)

T1_TRACKS = ['monza', 'barcelona', 'red_bull_ring']

all_raw = [f for f in (h_pkl + h_ld + s_pkl) if is_telemetry_file(f)]

tier_files = {'T1_human': [], 'T1_sac': [],
              'T2_new_human': [], 'T2_new_sac': [],
              'T3_new_human': [], 'T3_new_sac': [],
              'T4_new_human': [], 'T4_new_sac': []}

for f in all_raw:
    t, c = classify_file(f)
    suf = '_sac' if is_sac_file(f) else '_human'
    if t in T1_TRACKS and c == 'bmw_z4_gt3':
        tier_files['T1' + suf].append(f)
    elif t in T1_TRACKS and c == 'dallara_f317':
        tier_files['T2_new' + suf].append(f)
    elif t == 'barcelona' and c == 'ks_mazda_miata':
        tier_files['T3_new' + suf].append(f)
    elif t != 'unknown' and c != 'unknown':
        tier_files['T4_new' + suf].append(f)

print()
for key, files in tier_files.items():
    print(f"  {key:<20s}: {len(files):>5} dosya")
print(f"\n  Toplam: {sum(len(v) for v in tier_files.values())}")

## Adim 3: Format Kontrolu (T1 ile uyumluluk)

In [ ]:
print("=" * 65)
print("  ADIM 3: FORMAT KONTROLU")
print("=" * 65)

sample_files = tier_files['T2_new_human'][:1] or tier_files['T3_new_human'][:1]
if not sample_files:
    print("Yeni tier dosyasi bulunamadi")
else:
    sample = sample_files[0]
    print(f"\n  Ornek: {sample.name} ({sample.stat().st_size / 1024:.0f} KB)")
    
    with open(sample, 'rb') as f:
        data = pickle.load(f)
    
    if isinstance(data, dict) and 'states' in data:
        states = data['states']
        static_info = data.get('static_info', {})
        print(f"  states: shape={states.shape if hasattr(states,'shape') else '?'}")
        
        ch_names = None
        for key in ['obs_features', 'features', 'columns', 'channel_names']:
            if key in static_info:
                ch_names = list(static_info[key])
                print(f"  Kanal isimleri: '{key}' ({len(ch_names)} adet)")
                break
        
        if ch_names and T1_COLUMNS:
            common = set(ch_names) & set(T1_COLUMNS)
            print(f"  T1 ile ortak: {len(common)}/{len(T1_COLUMNS)}")
            if len(common) >= len(T1_COLUMNS) * 0.9:
                print("  -> T1 ile uyumlu!")
    else:
        print(f"  Tip: {type(data).__name__}")

## Adim 4: Donusum Fonksiyonu

In [ ]:
def convert_pkl_to_parquet(pkl_path, output_dir, prefix=""):
    try:
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        
        df = None
        
        if isinstance(data, dict) and 'states' in data:
            states = data['states']
            static_info = data.get('static_info', {})
            
            # Format A: states = list of dicts (stint PKL, 183 kanal)
            if isinstance(states, list) and len(states) > 0 and isinstance(states[0], dict):
                df = pd.DataFrame(states)
            
            # Format B: states = numpy array + static_info dict (raw_data PKL)
            elif isinstance(states, np.ndarray) and states.ndim == 2:
                ch_names = None
                if isinstance(static_info, dict):
                    for key in ['obs_features', 'features', 'columns', 'channel_names']:
                        if key in static_info:
                            ch_names = list(static_info[key])
                            break
                if ch_names and len(ch_names) == states.shape[1]:
                    df = pd.DataFrame(states, columns=ch_names)
                else:
                    df = pd.DataFrame(states)
        
        elif isinstance(data, pd.DataFrame):
            df = data
        
        if df is None or len(df) == 0:
            return None, "Donusturulemedi"
        
        t, c = classify_file(pkl_path)
        stem = pkl_path.stem.replace("raw_data", "").strip("_")
        if not stem:
            stem = pkl_path.parent.name
        out_name = f"{prefix}{t}_{c}_{stem}.parquet"
        out_path = output_dir / out_name
        df.to_parquet(out_path, index=False)
        return out_path, None
    except Exception as e:
        return None, str(e)

print("convert_pkl_to_parquet() tanimlandi (Format A + B)")

## Adim 5: T2 Donusum (Dallara F317 x 3 pist)
**Uzun surebilir**

In [ ]:
print("=" * 65)
print("  ADIM 5: T2 DONUSUM")
print("=" * 65)

T2_DIR = TIER_DIRS['T2']

t1_copied = 0
for f in TIER_DIRS['T1'].glob("*.parquet"):
    dst = T2_DIR / f.name
    if not dst.exists():
        copy2(f, dst)
        t1_copied += 1
print(f"  T1->T2 kopyalanan: {t1_copied}")

print(f"\n--- INSAN ({len(tier_files['T2_new_human'])}) ---")
ok, fail = 0, 0
for i, f in enumerate(tier_files['T2_new_human']):
    path, err = convert_pkl_to_parquet(f, T2_DIR)
    ok += 1 if path else 0
    fail += 0 if path else 1
    if (i+1) % 50 == 0 or i < 3 or i == len(tier_files['T2_new_human'])-1:
        print(f"  [{i+1}/{len(tier_files['T2_new_human'])}] ok={ok} fail={fail}")

print(f"\n--- SAC ({len(tier_files['T2_new_sac'])}) -> reference_sac/ ---")
ok_s, fail_s = 0, 0
for i, f in enumerate(tier_files['T2_new_sac']):
    path, err = convert_pkl_to_parquet(f, SAC_REF, prefix="sac_")
    ok_s += 1 if path else 0
    fail_s += 0 if path else 1
    if (i+1) % 500 == 0 or i < 3 or i == len(tier_files['T2_new_sac'])-1:
        print(f"  [{i+1}/{len(tier_files['T2_new_sac'])}] ok={ok_s} fail={fail_s}")

print(f"\n  T2: {len(list(T2_DIR.glob('*.parquet')))} parquet")

In [ ]:
# states[0] dict'inin yapısı
print(f"states[0] keys ({len(data['states'][0])}):")
for k, v in list(data['states'][0].items())[:15]:
    print(f"  '{k}': {type(v).__name__} = {v}" if not hasattr(v, '__len__') or isinstance(v, str) else f"  '{k}': {type(v).__name__}, len={len(v)}")

print(f"\nstatic_info: '{data['static_info'][:200]}'")

## Adim 6: T3 Donusum (+ Barcelona Mazda)

In [ ]:
print("=" * 65)
print("  ADIM 6: T3 DONUSUM")
print("=" * 65)

T3_DIR = TIER_DIRS['T3']
t2_copied = 0
for f in TIER_DIRS['T2'].glob("*.parquet"):
    dst = T3_DIR / f.name
    if not dst.exists():
        copy2(f, dst)
        t2_copied += 1
print(f"  T2->T3 kopyalanan: {t2_copied}")

print(f"\n--- INSAN ({len(tier_files['T3_new_human'])}) ---")
ok, fail = 0, 0
for i, f in enumerate(tier_files['T3_new_human']):
    path, err = convert_pkl_to_parquet(f, T3_DIR)
    ok += 1 if path else 0
    fail += 0 if path else 1
    if (i+1) % 100 == 0 or i < 3 or i == len(tier_files['T3_new_human'])-1:
        print(f"  [{i+1}/{len(tier_files['T3_new_human'])}] ok={ok} fail={fail}")

print(f"\n--- SAC ({len(tier_files['T3_new_sac'])}) -> reference_sac/ ---")
ok_s, fail_s = 0, 0
for i, f in enumerate(tier_files['T3_new_sac']):
    path, err = convert_pkl_to_parquet(f, SAC_REF, prefix="sac_")
    ok_s += 1 if path else 0
    fail_s += 0 if path else 1
    if (i+1) % 200 == 0 or i < 3 or i == len(tier_files['T3_new_sac'])-1:
        print(f"  [{i+1}/{len(tier_files['T3_new_sac'])}] ok={ok_s} fail={fail_s}")

print(f"\n  T3: {len(list(T3_DIR.glob('*.parquet')))} parquet")

## Adim 7: T4 Donusum (Indianapolis + kalan)

In [ ]:
print("=" * 65)
print("  ADIM 7: T4 DONUSUM")
print("=" * 65)

T4_DIR = TIER_DIRS['T4']
t3_copied = 0
for f in TIER_DIRS['T3'].glob("*.parquet"):
    dst = T4_DIR / f.name
    if not dst.exists():
        copy2(f, dst)
        t3_copied += 1
print(f"  T3->T4 kopyalanan: {t3_copied}")

print(f"\n--- INSAN ({len(tier_files['T4_new_human'])}) ---")
ok, fail = 0, 0
for i, f in enumerate(tier_files['T4_new_human']):
    path, err = convert_pkl_to_parquet(f, T4_DIR)
    ok += 1 if path else 0
    fail += 0 if path else 1
    if (i+1) % 50 == 0 or i < 3 or i == len(tier_files['T4_new_human'])-1:
        print(f"  [{i+1}/{len(tier_files['T4_new_human'])}] ok={ok} fail={fail}")

print(f"\n--- SAC ({len(tier_files['T4_new_sac'])}) -> reference_sac/ ---")
ok_s, fail_s = 0, 0
for i, f in enumerate(tier_files['T4_new_sac']):
    path, err = convert_pkl_to_parquet(f, SAC_REF, prefix="sac_")
    ok_s += 1 if path else 0
    fail_s += 0 if path else 1
    if (i+1) % 200 == 0 or i < 3 or i == len(tier_files['T4_new_sac'])-1:
        print(f"  [{i+1}/{len(tier_files['T4_new_sac'])}] ok={ok_s} fail={fail_s}")

print(f"\n  T4: {len(list(T4_DIR.glob('*.parquet')))} parquet")

## Adim 8: Dogrulama Raporu

In [ ]:
import shutil as sh

print("=" * 65)
print("  NB03 v2 DONUSUM RAPORU")
print("=" * 65)

for tier_name, tier_dir in TIER_DIRS.items():
    count = len(list(tier_dir.glob("*.parquet")))
    dist = {}
    for f in tier_dir.glob("*.parquet"):
        t, c = classify_file(f)
        k = f"{t} x {c}"
        dist[k] = dist.get(k, 0) + 1
    print(f"\n  {tier_name}: {count} parquet")
    for k in sorted(dist):
        print(f"    {k:<35s} {dist[k]:>5}")

sac_count = len(list(SAC_REF.glob("*.parquet")))
print(f"\n  SAC reference: {sac_count} parquet")

total, used, free = sh.disk_usage("D:\\")
print(f"  Disk: {free/1e9:.0f} GB bos / {total/1e9:.0f} GB toplam")
print(f"\n  SONRAKI: NB07 -> her tier icin viraj tespiti")

In [ ]:
from pathlib import Path

PROJECT = TRACK_ROOT
DATA = PROJECT / "data"

print("═" * 65)
print("  MEVCUT PİPELİNE ÇIKTILARI")
print("═" * 65)

# NB07 viraj referans dosyaları
for folder in ['features', 'processed', 'fingerprints']:
    base = DATA / folder
    if base.exists():
        all_files = list(base.rglob("*"))
        files = [f for f in all_files if f.is_file()]
        print(f"\n📂 data/{folder}/ ({len(files)} dosya):")
        for f in sorted(files)[:20]:
            size = f.stat().st_size / 1024
            print(f"  {f.relative_to(base)} ({size:.0f}KB)")
        if len(files) > 20:
            print(f"  ... +{len(files)-20} daha")

# Tier0 baseline - corner references var mı?
tier0 = DATA / "tier0_baseline"
if tier0.exists():
    t0_files = list(tier0.rglob("*"))
    t0_files = [f for f in t0_files if f.is_file()]
    print(f"\n📂 tier0_baseline/ ({len(t0_files)} dosya):")
    for f in sorted(t0_files)[:15]:
        print(f"  {f.relative_to(tier0)} ({f.stat().st_size/1024:.0f}KB)")

In [ ]:
import pandas as pd
from pathlib import Path

feat = (TRACK_ROOT / "data" / "features")

for track in ['barcelona', 'monza', 'red_bull_ring']:
    f = feat / f"driver_corner_matrix_{track}.parquet"
    if f.exists():
        df = pd.read_parquet(f)
        print(f"\n{track}:")
        print(f"  Shape: {df.shape}")
        print(f"  Kolonlar: {list(df.columns)[:8]}...")
        if 'driver' in df.columns:
            print(f"  Sürücüler: {sorted(df['driver'].unique())}")
        elif 'driver_id' in df.columns:
            print(f"  Sürücüler: {sorted(df['driver_id'].unique())}")
        print(f"  İlk 3 satır:\n{df.head(3)}")